# Compact teacher redesign notebook — corrected oracle benchmark

This notebook builds single-label oracle teacher candidates for equities and fixed income at 3M and 6M horizons, screens them, and runs a quick student benchmark.

In [ ]:
from pathlib import Path
import warnings
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, log_loss

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

WORKDIR = Path.cwd()
MODEL_PATH = WORKDIR / "modeling_panels" / "model_panel_start_1970_01_31.parquet"

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing modeling panel: {MODEL_PATH}")

panel = pd.read_parquet(MODEL_PATH).copy()
panel["date"] = pd.to_datetime(panel["date"])
panel = panel.sort_values("date").reset_index(drop=True)

print("Loaded:", MODEL_PATH)
print("Shape:", panel.shape)

REQUIRED_COLS = [
    "date",
    "sp500_fwd_excess_3m", "sp500_fwd_excess_6m",
    "bond_fwd_excess_3m", "bond_fwd_excess_6m",
    "sprtrn_sp500",
    "agg_ret",
    "credit_spread_baa_aaa",
    "GS10",
]

missing = [c for c in REQUIRED_COLS if c not in panel.columns]
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

ALIASES = {
    "eq_fwd_excess_3m": "sp500_fwd_excess_3m",
    "eq_fwd_excess_6m": "sp500_fwd_excess_6m",
    "fi_fwd_excess_3m": "bond_fwd_excess_3m",
    "fi_fwd_excess_6m": "bond_fwd_excess_6m",
    "eq_ret_1m": "sprtrn_sp500",
    "fi_ret_1m": "agg_ret",
    "spread_proxy": "credit_spread_baa_aaa",
    "yield_proxy": "GS10",
}

for c in REQUIRED_COLS:
    if c != "date":
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

for alias, col in ALIASES.items():
    panel[alias] = pd.to_numeric(panel[col], errors="coerce")


In [ ]:
def future_realized_vol(ret, horizon):
    ret = pd.to_numeric(ret, errors="coerce")
    out = pd.Series(index=ret.index, dtype=float)
    for i in range(len(ret)):
        win = ret.iloc[i + 1:i + 1 + horizon]
        if len(win) < horizon or win.isna().any():
            out.iloc[i] = np.nan
        else:
            out.iloc[i] = win.std(ddof=1)
    return out

def future_downside_freq(ret, horizon):
    ret = pd.to_numeric(ret, errors="coerce")
    out = pd.Series(index=ret.index, dtype=float)
    for i in range(len(ret)):
        win = ret.iloc[i + 1:i + 1 + horizon]
        if len(win) < horizon or win.isna().any():
            out.iloc[i] = np.nan
        else:
            out.iloc[i] = (win < 0).mean()
    return out

def future_abs_change(x, horizon):
    x = pd.to_numeric(x, errors="coerce")
    out = pd.Series(index=x.index, dtype=float)
    for i in range(len(x)):
        j = i + horizon
        if j >= len(x) or pd.isna(x.iloc[i]) or pd.isna(x.iloc[j]):
            out.iloc[i] = np.nan
        else:
            out.iloc[i] = abs(x.iloc[j] - x.iloc[i])
    return out

def full_sample_median_threshold_state(score):
    score = pd.to_numeric(score, errors="coerce")
    state = pd.Series(index=score.index, dtype=float)
    valid = score.notna()
    if valid.sum() == 0:
        state[:] = np.nan
        return state
    thr = score.loc[valid].median()
    state[:] = np.nan
    state.loc[valid] = (score.loc[valid] > thr).astype(float)
    return state

def smooth_majority_vote(binary_state, window=3):
    s = pd.to_numeric(binary_state, errors="coerce")
    out = pd.Series(index=s.index, dtype=float)
    arr = s.to_numpy(dtype=float)
    for i in range(len(arr)):
        lo = max(0, i - window + 1)
        win = arr[lo:i + 1]
        win = win[np.isfinite(win)]
        out.iloc[i] = np.nan if len(win) == 0 else (1.0 if win.mean() >= 0.5 else 0.0)
    return out

def enforce_min_run_length(binary_state, min_run=2):
    s = pd.to_numeric(binary_state, errors="coerce")
    vals = s.to_numpy(dtype=float)
    out = vals.copy()
    n = len(vals)
    i = 0
    while i < n:
        if not np.isfinite(out[i]):
            i += 1
            continue
        j = i + 1
        while j < n and np.isfinite(out[j]) and out[j] == out[i]:
            j += 1
        run_len = j - i
        if run_len < min_run:
            prev_val = np.nan
            k = i - 1
            while k >= 0:
                if np.isfinite(out[k]):
                    prev_val = out[k]
                    break
                k -= 1
            next_val = np.nan
            k = j
            while k < n:
                if np.isfinite(out[k]):
                    next_val = out[k]
                    break
                k += 1
            fill = prev_val if np.isfinite(prev_val) else next_val
            if np.isfinite(fill):
                out[i:j] = fill
                back = i - 1
                while back >= 0 and np.isfinite(out[back]) and out[back] == fill:
                    back -= 1
                i = back + 1
                continue
        i = j
    return pd.Series(out, index=s.index)

def persistence_variant(raw_state, min_run):
    return enforce_min_run_length(smooth_majority_vote(raw_state, window=3), min_run=min_run)

def class_balance_stats(s):
    z = pd.to_numeric(s, errors="coerce").dropna().astype(int)
    if len(z) == 0:
        return {"n_obs": 0, "count_0": 0, "count_1": 0, "minority_share": np.nan}
    cnt = z.value_counts().to_dict()
    c0, c1 = cnt.get(0, 0), cnt.get(1, 0)
    return {
        "n_obs": len(z),
        "count_0": c0,
        "count_1": c1,
        "minority_share": min(c0, c1) / len(z),
    }

def persistence_stats(s):
    z = pd.to_numeric(s, errors="coerce").dropna().astype(int)
    if len(z) < 2:
        return {"switch_rate": np.nan, "same_as_lag_share": np.nan, "avg_run_length": np.nan}
    switches = (z != z.shift(1)).fillna(False)
    switch_rate = switches.iloc[1:].mean()
    same_as_lag_share = (z.iloc[1:].to_numpy() == z.iloc[:-1].to_numpy()).mean()
    runs = []
    cur = z.iloc[0]
    run = 1
    for val in z.iloc[1:]:
        if val == cur:
            run += 1
        else:
            runs.append(run)
            cur = val
            run = 1
    runs.append(run)
    return {
        "switch_rate": switch_rate,
        "same_as_lag_share": same_as_lag_share,
        "avg_run_length": float(np.mean(runs)),
    }

def train_test_balance_stats(s, test_fraction=0.25):
    z = pd.to_numeric(s, errors="coerce").dropna().astype(int)
    if len(z) == 0:
        return {}
    split = int(len(z) * (1 - test_fraction))
    tr = z.iloc[:split]
    te = z.iloc[split:]
    def one_part(x, prefix):
        cnt = x.value_counts().to_dict()
        c0, c1 = cnt.get(0, 0), cnt.get(1, 0)
        n = len(x)
        return {
            f"{prefix}_n": n,
            f"{prefix}_count_0": c0,
            f"{prefix}_count_1": c1,
            f"{prefix}_minority_share": min(c0, c1) / n if n else np.nan,
            f"{prefix}_single_class": int((c0 == 0) or (c1 == 0)),
        }
    out = {}
    out.update(one_part(tr, "train"))
    out.update(one_part(te, "test"))
    return out

def economic_separation(df, teacher_col, asset, horizon):
    if asset == "equity":
        ex_col = f"eq_fwd_excess_{horizon}m"
        vol_col = f"eq_fwd_vol_{horizon}m"
        down_col = f"eq_fwd_downside_{horizon}m"
        extra_cols = []
    else:
        ex_col = f"fi_fwd_excess_{horizon}m"
        vol_col = f"fi_fwd_vol_{horizon}m"
        down_col = f"fi_fwd_downside_{horizon}m"
        extra_cols = [f"future_abs_yield_change_{horizon}m", f"future_abs_spread_change_{horizon}m"]
    out = {}
    for state in [0, 1]:
        mask = df[teacher_col] == state
        out[f"avg_fwd_excess_state{state}"] = df.loc[mask, ex_col].mean()
        out[f"avg_fwd_vol_state{state}"] = df.loc[mask, vol_col].mean()
        out[f"avg_fwd_downside_state{state}"] = df.loc[mask, down_col].mean()
        for col in extra_cols:
            out[f"avg_{col}_state{state}"] = df.loc[mask, col].mean()
    out["gap_fwd_excess_good_minus_bad"] = out.get("avg_fwd_excess_state1", np.nan) - out.get("avg_fwd_excess_state0", np.nan)
    out["gap_fwd_vol_good_minus_bad"] = out.get("avg_fwd_vol_state1", np.nan) - out.get("avg_fwd_vol_state0", np.nan)
    out["gap_fwd_downside_good_minus_bad"] = out.get("avg_fwd_downside_state1", np.nan) - out.get("avg_fwd_downside_state0", np.nan)
    return out

def run_lengths(s):
    z = pd.to_numeric(s, errors="coerce").dropna().astype(int)
    if len(z) == 0:
        return []
    runs = []
    cur = z.iloc[0]
    run = 1
    for val in z.iloc[1:]:
        if val == cur:
            run += 1
        else:
            runs.append(run)
            cur = val
            run = 1
    runs.append(run)
    return runs


In [ ]:
panel["eq_fwd_vol_3m"] = future_realized_vol(panel["eq_ret_1m"], 3)
panel["eq_fwd_vol_6m"] = future_realized_vol(panel["eq_ret_1m"], 6)
panel["eq_fwd_downside_3m"] = future_downside_freq(panel["eq_ret_1m"], 3)
panel["eq_fwd_downside_6m"] = future_downside_freq(panel["eq_ret_1m"], 6)

panel["fi_fwd_vol_3m"] = future_realized_vol(panel["fi_ret_1m"], 3)
panel["fi_fwd_vol_6m"] = future_realized_vol(panel["fi_ret_1m"], 6)
panel["fi_fwd_downside_3m"] = future_downside_freq(panel["fi_ret_1m"], 3)
panel["fi_fwd_downside_6m"] = future_downside_freq(panel["fi_ret_1m"], 6)

panel["future_abs_yield_change_3m"] = future_abs_change(panel["yield_proxy"], 3)
panel["future_abs_yield_change_6m"] = future_abs_change(panel["yield_proxy"], 6)
panel["future_abs_spread_change_3m"] = future_abs_change(panel["spread_proxy"], 3)
panel["future_abs_spread_change_6m"] = future_abs_change(panel["spread_proxy"], 6)

LAMBDA_EQ_VOL = 0.5
LAMBDA_EQ_DOWN = 0.75
LAMBDA_FI_VOL = 0.5
LAMBDA_FI_SPREAD = 0.5
LAMBDA_FI_YIELD = 0.5

for h in [3, 6]:
    panel[f"eq_score_A_{h}m"] = panel[f"eq_fwd_excess_{h}m"]
    panel[f"eq_score_B_{h}m"] = panel[f"eq_fwd_excess_{h}m"] - LAMBDA_EQ_VOL * panel[f"eq_fwd_vol_{h}m"]
    panel[f"eq_score_C_{h}m"] = panel[f"eq_fwd_excess_{h}m"] - LAMBDA_EQ_DOWN * panel[f"eq_fwd_downside_{h}m"]

    panel[f"fi_score_A_{h}m"] = panel[f"fi_fwd_excess_{h}m"]
    panel[f"fi_score_B_{h}m"] = panel[f"fi_fwd_excess_{h}m"] - LAMBDA_FI_VOL * panel[f"fi_fwd_vol_{h}m"]
    panel[f"fi_score_C_{h}m"] = (
        panel[f"fi_fwd_excess_{h}m"]
        - LAMBDA_FI_SPREAD * panel[f"future_abs_spread_change_{h}m"]
        - LAMBDA_FI_YIELD * panel[f"future_abs_yield_change_{h}m"]
    )


In [ ]:
TEACHERS_RAW = {}
TEACHERS_FINAL = {}

def register_raw_teacher(name, series, asset, horizon, family, score_col):
    TEACHERS_RAW[name] = {
        "series": series,
        "asset": asset,
        "horizon_m": horizon,
        "family": family,
        "score_col": score_col,
    }

def register_final_teacher(name, series, asset, horizon, family, score_col, min_run):
    TEACHERS_FINAL[name] = {
        "series": series,
        "asset": asset,
        "horizon_m": horizon,
        "family": family,
        "score_col": score_col,
        "min_run": min_run,
    }

for asset_prefix, asset_name in [("eq", "equity"), ("fi", "fixed_income")]:
    for family in ["A", "B", "C"]:
        for h in [3, 6]:
            score_col = f"{asset_prefix}_score_{family}_{h}m"
            raw_name = f"{asset_prefix}_raw_{family}_{h}m"
            raw_state = full_sample_median_threshold_state(panel[score_col])
            panel[raw_name] = raw_state
            register_raw_teacher(raw_name, raw_state, asset_name, h, family, score_col)

for raw_name, meta in TEACHERS_RAW.items():
    for min_run in [2, 3]:
        final_name = raw_name.replace("_raw_", f"_mr{min_run}_")
        final_series = persistence_variant(meta["series"], min_run=min_run)
        panel[final_name] = final_series
        register_final_teacher(
            final_name,
            final_series,
            meta["asset"],
            meta["horizon_m"],
            meta["family"],
            meta["score_col"],
            min_run,
        )

print("Raw teachers:", len(TEACHERS_RAW))
print("Final teachers:", len(TEACHERS_FINAL))


In [ ]:
check_rows = []
for name in TEACHERS_FINAL:
    runs = run_lengths(panel[name])
    check_rows.append({
        "teacher": name,
        "min_run": TEACHERS_FINAL[name]["min_run"],
        "min_observed_run": min(runs) if len(runs) else np.nan,
        "n_runs": len(runs),
    })
run_check = pd.DataFrame(check_rows).sort_values("teacher").reset_index(drop=True)
print(run_check.to_string(index=False))


In [ ]:
diag_rows = []
for name, meta in TEACHERS_FINAL.items():
    row = {
        "teacher": name,
        "asset": meta["asset"],
        "horizon_m": meta["horizon_m"],
        "family": meta["family"],
        "min_run": meta["min_run"],
    }
    row.update(class_balance_stats(panel[name]))
    row.update(train_test_balance_stats(panel[name], test_fraction=0.25))
    row.update(persistence_stats(panel[name]))
    row.update(economic_separation(panel, name, meta["asset"], meta["horizon_m"]))
    diag_rows.append(row)

diag = pd.DataFrame(diag_rows).sort_values(["asset", "horizon_m", "family", "min_run"]).reset_index(drop=True)
diag["survives_first_pass"] = (
    (diag["minority_share"] >= 0.20)
    & (diag["test_single_class"] == 0)
    & diag["switch_rate"].between(0.05, 0.45, inclusive="both")
    & (diag["same_as_lag_share"] < 0.95)
    & (diag["avg_run_length"] >= 2.0)
    & (diag["gap_fwd_excess_good_minus_bad"] > 0)
)

diag_view_cols = [
    "teacher", "asset", "horizon_m", "family", "min_run",
    "minority_share", "switch_rate", "avg_run_length",
    "gap_fwd_excess_good_minus_bad", "gap_fwd_vol_good_minus_bad",
    "gap_fwd_downside_good_minus_bad", "survives_first_pass"
]
print(diag[diag_view_cols].to_string(index=False))

pair_rows = []
for asset_prefix in ["eq", "fi"]:
    for family in ["A", "B", "C"]:
        for h in [3, 6]:
            c2 = f"{asset_prefix}_mr2_{family}_{h}m"
            c3 = f"{asset_prefix}_mr3_{family}_{h}m"
            mask = panel[[c2, c3]].notna().all(axis=1)
            same_share = (panel.loc[mask, c2] == panel.loc[mask, c3]).mean() if mask.any() else np.nan
            pair_rows.append({
                "teacher_pair": f"{asset_prefix}_{family}_{h}m",
                "same_share_mr2_vs_mr3": same_share,
            })
pair_df = pd.DataFrame(pair_rows).sort_values("teacher_pair").reset_index(drop=True)
print("\nMR2 vs MR3 overlap")
print(pair_df.to_string(index=False))

survivors = diag.loc[diag["survives_first_pass"]].copy()


In [ ]:
TEST_FRACTION = 0.25
CV_SPLITS = 5
C_GRID = np.logspace(-4, 2, 15)
MAX_ITER = 5000
RANDOM_STATE = 42

def build_l2_model(C):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l2",
            C=C,
            solver="lbfgs",
            max_iter=MAX_ITER,
            random_state=RANDOM_STATE,
        )),
    ])

def get_feature_cols(df, target_col):
    exclude_cols = {"date", target_col}
    for c in df.columns:
        c_low = c.lower()
        if (
            ("label" in c_low)
            or ("teacher" in c_low)
            or ("_raw_" in c_low)
            or ("_mr2_" in c_low)
            or ("_mr3_" in c_low)
            or ("score_" in c_low)
            or ("fwd_" in c_low)
            or ("forward" in c_low)
        ):
            exclude_cols.add(c)
    return [c for c in df.columns if c not in exclude_cols]

def fit_best_l2_timeseries_cv(X_tr, y_tr):
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
    rows = []
    for C in C_GRID:
        fold_losses = []
        for tr_idx, va_idx in tscv.split(X_tr):
            X_tr_cv = X_tr.iloc[tr_idx]
            y_tr_cv = y_tr.iloc[tr_idx]
            X_va_cv = X_tr.iloc[va_idx]
            y_va_cv = y_tr.iloc[va_idx]
            if y_tr_cv.nunique() < 2 or y_va_cv.nunique() < 2:
                continue
            model = build_l2_model(C)
            model.fit(X_tr_cv, y_tr_cv)
            proba = model.predict_proba(X_va_cv)
            fold_losses.append(log_loss(y_va_cv, proba, labels=[0, 1]))
        rows.append({"C": C, "cv_log_loss": np.mean(fold_losses) if fold_losses else np.nan})
    cv_df = pd.DataFrame(rows).dropna().sort_values(["cv_log_loss", "C"]).reset_index(drop=True)
    best_C = float(cv_df.iloc[0]["C"])
    final_model = build_l2_model(best_C)
    final_model.fit(X_tr, y_tr)
    return final_model, best_C

result_rows = []
for target_col in survivors["teacher"].tolist():
    feature_cols = get_feature_cols(panel, target_col)
    work = panel[["date", target_col] + feature_cols].copy()
    work = work[work[target_col].notna()].copy()
    work[target_col] = work[target_col].astype(int)
    if work[target_col].nunique() < 2:
        continue
    split_idx = int(len(work) * (1 - TEST_FRACTION))
    train_df = work.iloc[:split_idx].copy()
    test_df = work.iloc[split_idx:].copy()
    if train_df[target_col].nunique() < 2 or test_df[target_col].nunique() < 2:
        continue
    X_tr = train_df[feature_cols]
    y_tr = train_df[target_col]
    X_te = test_df[feature_cols]
    y_te = test_df[target_col]
    model, best_C = fit_best_l2_timeseries_cv(X_tr, y_tr)
    proba_te = model.predict_proba(X_te)
    pred_te = model.predict(X_te)
    y_te_prev = test_df[target_col].shift(1)
    valid_base = y_te_prev.notna()
    y_te_base = y_te.loc[valid_base]
    pred_base = y_te_prev.loc[valid_base].astype(int)
    p1 = y_tr.mean()
    p0 = 1.0 - p1
    proba_freq = np.column_stack([np.full(len(y_te), p0), np.full(len(y_te), p1)])
    pred_freq = np.where(p1 >= 0.5, 1, 0)
    meta = survivors.loc[survivors["teacher"] == target_col].iloc[0]
    result_rows.append({
        "target": target_col,
        "asset": meta["asset"],
        "horizon_m": int(meta["horizon_m"]),
        "family": meta["family"],
        "min_run": int(meta["min_run"]),
        "best_C": best_C,
        "l2_accuracy": accuracy_score(y_te, pred_te),
        "l2_balanced_accuracy": balanced_accuracy_score(y_te, pred_te),
        "l2_log_loss": log_loss(y_te, proba_te, labels=[0, 1]),
        "persist_accuracy": accuracy_score(y_te_base, pred_base),
        "freq_accuracy": accuracy_score(y_te, np.full(len(y_te), pred_freq)),
        "freq_log_loss": log_loss(y_te, proba_freq, labels=[0, 1]),
        "l2_minus_persist_acc": accuracy_score(y_te, pred_te) - accuracy_score(y_te_base, pred_base),
        "l2_minus_freq_acc": accuracy_score(y_te, pred_te) - accuracy_score(y_te, np.full(len(y_te), pred_freq)),
        "l2_minus_freq_logloss": log_loss(y_te, proba_freq, labels=[0, 1]) - log_loss(y_te, proba_te, labels=[0, 1]),
    })

student_test_results = pd.DataFrame(result_rows).sort_values(["l2_log_loss", "l2_balanced_accuracy"], ascending=[True, False]).reset_index(drop=True)
print(student_test_results.to_string(index=False))

print("\nAverage by asset and horizon")
summary_by_asset_horizon = (
    student_test_results
    .groupby(["asset", "horizon_m"], as_index=False)[["l2_accuracy", "l2_balanced_accuracy", "l2_log_loss", "l2_minus_persist_acc", "l2_minus_freq_logloss"]]
    .mean()
    .sort_values(["asset", "horizon_m"])
    .reset_index(drop=True)
)
print(summary_by_asset_horizon.to_string(index=False))


In [ ]:
def build_index_from_returns(ret, start=100.0):
    ret = pd.to_numeric(ret, errors="coerce")
    idx = pd.Series(index=ret.index, dtype=float)
    level = start
    for i, r in enumerate(ret):
        if pd.isna(r):
            idx.iloc[i] = np.nan
        else:
            level *= (1.0 + r)
            idx.iloc[i] = level
    return idx

def plot_teacher_path_grid(df, specs, ncols=2, figsize_per_row=4.4):
    n = len(specs)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, figsize_per_row * nrows), squeeze=False)
    axes = axes.flatten()

    for ax, (teacher_col, path_col, title) in zip(axes, specs):
        sub = df[["date", teacher_col, path_col]].copy()
        sub = sub[sub[teacher_col].notna() & sub[path_col].notna()].copy()

        if sub.empty:
            ax.set_title(f"{title}\n(no usable data)")
            ax.axis("off")
            continue

        ax.plot(sub["date"], sub[path_col], linewidth=1.6)

        state = sub[teacher_col].to_numpy()
        dates = sub["date"].to_numpy()
        start = 0

        for i in range(1, len(sub) + 1):
            boundary = (i == len(sub)) or (state[i] != state[start])
            if boundary:
                x0 = dates[start]
                x1 = dates[i - 1]
                if state[start] == 1:
                    ax.axvspan(x0, x1, alpha=0.18)
                else:
                    ax.axvspan(x0, x1, alpha=0.06)
                start = i

        ax.set_title(title)
        ax.set_xlabel("date")
        ax.set_ylabel(path_col)

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

panel["sp500_path_from_ret"] = build_index_from_returns(panel["eq_ret_1m"], start=100.0)
panel["agg_path_from_ret"] = build_index_from_returns(panel["fi_ret_1m"], start=100.0)

path_specs = [
    ("eq_mr2_B_3m", "sp500_path_from_ret", "Equity B 3M | final state against SP500 path"),
    ("eq_mr2_C_6m", "sp500_path_from_ret", "Equity C 6M | final state against SP500 path"),
    ("fi_mr2_B_3m", "agg_path_from_ret", "Fixed income B 3M | final state against Agg path"),
    ("fi_mr2_C_6m", "agg_path_from_ret", "Fixed income C 6M | final state against Agg path"),
]

plot_teacher_path_grid(panel, path_specs, ncols=2)
